# Deepfake Sampling & Visualization Standalone
This notebook mounts Google Drive, accesses the HiDF dataset, and generates side-by-side visualizations of **Real Source Images** and their **Generated Deepfake** counterparts.

**Convention:** In the HiDF dataset, a fake image named `ID1_ID2.jpg` is a composite of the real images `ID1.jpg` (Base) and `ID2.jpg` (Swap).

In [ ]:
import os
import random
import zipfile
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive
from tqdm.auto import tqdm

## 1. Setup & Path Configuration

In [ ]:
# Change these paths if your dataset is stored elsewhere
BASE_PATH = '/content'
DRIVE_MOUNT = '/content/drive'
PROJECT_FOLDER = DRIVE_MOUNT + '/MyDrive/DataMining/project_dataset'

REAL_DIR = BASE_PATH + '/Real-img'
FAKE_DIR = BASE_PATH + '/Image'

# Mount Google Drive
if not os.path.ismount(DRIVE_MOUNT):
    drive.mount(DRIVE_MOUNT)

def quick_extract(zip_name, target_path):
    if not os.path.exists(target_path):
        zip_path = os.path.join(PROJECT_FOLDER, zip_name)
        print(f"Extracting {zip_name} to {target_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(BASE_PATH)
    else:
        print(f"{target_path} already exists.")

# Extract datasets
quick_extract('Real-img.zip', REAL_DIR)
quick_extract('Fake-img.zip', FAKE_DIR)

## 2. Sample and Visualize Pairs

In [ ]:
def display_deepfake_ancestry(num_samples=3):
    fake_files = [f for f in os.listdir(FAKE_DIR) if f.endswith('.jpg')]
    samples = random.sample(fake_files, num_samples)
    
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = [axes] # Handle single sample indexing

    for i, fake_file in enumerate(samples):
        # Parse IDs
        parts = fake_file.split('.')[0].split('_')
        if len(parts) < 2: continue
        base_id, swap_id = parts[0], parts[1]
        
        # Load Images
        img_fake = Image.open(os.path.join(FAKE_DIR, fake_file))
        
        base_img_path = os.path.join(REAL_DIR, f"{base_id}.jpg")
        swap_img_path = os.path.join(REAL_DIR, f"{swap_id}.jpg")
        
        # Source A (Base)
        if os.path.exists(base_img_path):
            axes[i][0].imshow(Image.open(base_img_path))
        axes[i][0].set_title(f"Real Source A (Base: {base_id})")
        axes[i][0].axis('off')
        
        # Source B (Swap)
        if os.path.exists(swap_img_path):
            axes[i][1].imshow(Image.open(swap_img_path))
        axes[i][1].set_title(f"Real Source B (Swap: {swap_id})")
        axes[i][1].axis('off')
        
        # Result (Fake)
        axes[i][2].imshow(img_fake)
        axes[i][2].set_title("Generated Deepfake")
        axes[i][2].axis('off')

    plt.tight_layout()
    plt.show()

# Run the visualization
display_deepfake_ancestry(num_samples=3)